# Forensic self-descriptions: family structure, and structure *inside* a family

Each image is represented by its **forensic self-description** Φ = {φ₁ … φ_K} (Nguyen, Azizpour
& Stamm, CVPR 2025, [arXiv 2503.21003](https://arxiv.org/abs/2503.21003)). Φ is the set of
linear filters that best model the image's own forensic residuals: K residuals from a
real-only residual extractor (FRE), each modelled from its B×B neighbourhood across L scales.
Dimension = K·(B²−1). It is a *description* fitted per image, not a discriminative embedding:
no generator label is used anywhere in producing it.

Two questions, answered side by side on the **raw** descriptions and on the **family-centred**
ones (each family's mean Φ subtracted, then one PCA on the pooled remainder of the fake
families; granularity *g* = number of components kept; DECISIONS.md D-007):

1. Does Φ cluster by generator family? (raw)
2. Once the family is removed, do generators *of the same family* still separate, and does that
   hold as *g* grows? This is the precondition for the hypothesis that new, time-specific signals
   appear as a family evolves. Release date is shown as colour/order only. **Relating
   components to release date is deferred** to a later notebook.

Conclusions are checked across K, B, L and the authors' released FRE in §6.

**This notebook only loads.** Descriptions are produced by `selfdesc/` (see `selfdesc/README.md`):

```sh
python imports/build_manifest.py --out experiments/selfdesc-sample/manifest.csv
python -m selfdesc.sweep --manifest experiments/selfdesc-sample/manifest.csv \
    --out experiments/selfdesc-sample --workers 10
```

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from selfdesc import analysis as A

ROOT = Path.cwd()
EXP = ROOT / "experiments" / "selfdesc-sample"   # on the server: experiments/selfdesc-36k
CONFIG = "K8_M11_B11_L3"                          # the paper's setting; §6 sweeps the rest
SCALING = "robust"                                # see analysis.scale; §6 also tries the others
N_PERM = 500

# Palette: the reference categorical slots; sequential = one hue light -> dark.
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
CONTEXT, NULLBAND, INK, MUTED = "#d9d8d4", "#e4e3df", "#0b0b0b", "#52514e"
DATE_CMAP = LinearSegmentedColormap.from_list("date", ["#cfe2f8", BLUE, "#0b2a55"])
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.titlesize": 10, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
                     "legend.frameon": False})

print("available configs:", A.configs(EXP))
index, X, diag, meta = A.load(EXP, CONFIG)
fake = (index.label == 1).to_numpy()
fam = index.generator_family.to_numpy()
print(f"{CONFIG}: {X.shape[0]} images x {X.shape[1]} dims "
      f"(K={meta['K']}, M={meta['M']}, B={meta['B']}, L={meta['L']}), crop policy "
      f"{meta['crop_policy']}, FRE sha256 {meta['fre_sha256'][:12]}..., "
      f"lambda={meta['fre_meta']['lambda']:.2f}, commit {str(meta['git_commit'])[:8]}")

## 0. What is linked to each description

One row per image in `images.parquet`, aligned to every config's `descriptions.npz` (the loader
asserts the alignment). Format is recorded twice: `source_format` is the container the image
arrived in; `stored_format` is what every crop was actually cut from (all images were
normalised to JPEG q95 by `imports/sample/imaging.prepare_image`). `source_format_basis` says
where `source_format` came from. LAION's container was never recorded by the importer, so it
stays `unknown`.

In [ ]:
cols = ["image_id", "label", "generator", "generator_family", "release_date", "origin_dataset",
        "split", "source_width", "source_height", "source_format", "source_format_basis",
        "stored_format", "crop_policy"]
display(index[cols].sample(6, random_state=0))
display(pd.crosstab(index.generator_family, index.source_format).loc[A.FAMILY_ORDER])

## 1. QC: fit conditioning

The K residuals share most of their content (the FRE filters differ only by the diversity the
training objective buys), so the joint fit is ill-conditioned and Φ can carry large
cancelling coefficients. ‖Φ‖ is the symptom. The same happens with the authors' released FRE.
Heavy tails here are why the default scaling is median/IQR (`analysis.scale`), and why §6
re-runs everything under the other scalings.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, col, title in [(axes[0], "phi_norm", "‖Φ‖ per image"),
                       (axes[1], "fit_mse", "model error (mean squared, all scales)")]:
    data = [diag[col][index.generator_family == f] for f in A.FAMILY_ORDER]
    ax.boxplot(data, orientation="horizontal", widths=0.6, flierprops=dict(markersize=3, alpha=0.5),
               medianprops=dict(color=BLUE, linewidth=2))
    ax.set_yticks(range(1, len(A.FAMILY_ORDER) + 1), A.FAMILY_ORDER)
    ax.set_xscale("log")
    ax.set_title(title)
plt.tight_layout()
plt.show()

worst = diag.assign(generator=index.generator, origin=index.origin_dataset) \
            .nlargest(5, "phi_norm")[["image_id", "origin", "generator", "phi_norm", "fit_mse"]]
display(worst)

## 2. The two views

* **raw**: Φ after per-feature scaling, all images.
* **family-centred**: raw minus each family's mean (authentic counts as its own group), then one
  PCA **fitted on the fake families only**; authentic is projected onto those axes, not used to
  fit them.

Left: how much variance each view's leading components hold. The raw PCA is fitted on the same
fake rows, so the two curves differ only by the removed family means.

In [ ]:
Z = A.scale(X, SCALING)
Zc = A.family_centre(Z, fam)
P_c, pca_c = A.centred_pca(Zc, fake, n_components=100)
P_r, pca_r = A.centred_pca(Z - Z[fake].mean(0), fake, n_components=100)   # raw, same fit rows

between = 1 - (Zc[fake] ** 2).sum() / ((Z[fake] - Z[fake].mean(0)) ** 2).sum()
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(np.arange(1, 101), np.cumsum(pca_r.explained_variance_ratio_), color=ORANGE, lw=2)
ax.plot(np.arange(1, 101), np.cumsum(pca_c.explained_variance_ratio_), color=BLUE, lw=2)
ax.text(100, np.cumsum(pca_r.explained_variance_ratio_)[-1], " raw", color=INK, va="center")
ax.text(100, np.cumsum(pca_c.explained_variance_ratio_)[-1], " centred", color=INK, va="center")
for g in A.GRANULARITIES:
    ax.axvline(g, color=CONTEXT, lw=0.8, zorder=0)
ax.set_xscale("log"); ax.set_xlabel("components kept (g)"); ax.set_ylabel("cumulative variance")
ax.set_title("variance captured by the leading components (fake rows)")
plt.tight_layout(); plt.show()
print(f"share of fake-row variance that is between families: {between:.1%}")

## 3. Embeddings, raw vs family-centred

t-SNE on each view's first 50 PCs, one row per family, the family in blue over everything else in
grey. If Φ carries family structure, the left column shows compact blue regions. Centring removes
each family's mean by construction, so on the right the families should overlap. Whatever
spread remains there is within-family structure, which §5 measures.

In [ ]:
emb_r = TSNE(perplexity=30, random_state=0, init="pca").fit_transform(P_r[:, :50])
emb_c = TSNE(perplexity=30, random_state=0, init="pca").fit_transform(P_c[:, :50])

fig, axes = plt.subplots(len(A.FAMILY_ORDER), 2, figsize=(7.2, 2.6 * len(A.FAMILY_ORDER)))
for row, f in enumerate(A.FAMILY_ORDER):
    sel = fam == f
    for col, (emb, view) in enumerate([(emb_r, "raw"), (emb_c, "family-centred")]):
        ax = axes[row, col]
        ax.scatter(*emb[~sel].T, s=6, color=CONTEXT, linewidths=0)
        ax.scatter(*emb[sel].T, s=10, color=BLUE, linewidths=0.4, edgecolors="white")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{f} (n={sel.sum()}): {view}" if row == 0 else f"{f} (n={sel.sum()})")
plt.tight_layout(); plt.show()

## 4. Family structure and its confounds (raw | centred)

Balanced leave-one-out 5-NN accuracy (chance = 1/levels), its permutation null (95th
percentile), KMeans ARI/NMI (k = number of levels) and Calinski–Harabasz. Computed on the
first 50 PCs of each view.

The confound rows matter as much as the family rows. In AI-GenBench each generator comes from one
or two source datasets with a characteristic container mix (see the crosstab in §0), so
"clusters by generator" and "clusters by dataset / format / native size" cannot be told apart
from accuracy alone. If origin_dataset separates as well as generator does, the description
may be tracking the dataset's processing history rather than the generator.

In [ ]:
def table(rows):
    df = pd.DataFrame(rows).set_index(["question", "view"])
    keep = ["n", "levels", "chance", "knn_acc", "null_q95", "p", "ari", "nmi", "calinski_harabasz"]
    return df[[c for c in keep if c in df]].round(3)

size = A.size_bucket(index).to_numpy()
questions = [
    ("family (all images)", np.ones(len(index), bool), fam),
    ("family (fakes only)", fake, fam),
    ("real vs fake", np.ones(len(index), bool), index.label.to_numpy()),
    ("generator (fakes)", fake, index.generator.to_numpy()),
    ("origin_dataset (fakes)", fake, index.origin_dataset.to_numpy()),
    ("source_format (fakes)", fake, index.source_format.to_numpy()),
    ("native size bucket (fakes)", fake, size),
    ("origin_dataset (authentic)", ~fake, index.origin_dataset.to_numpy()),
]
rows = []
for q, sel, lab in questions:
    for view, P in [("raw", P_r), ("centred", P_c)]:
        rows.append(dict(question=q, view=view, **A.separability(P[sel, :50], lab[sel],
                                                                 n_perm=N_PERM)))
summary = table(rows)
summary

Generators share origin datasets unevenly. The table below counts, per generator, how many
fakes come from each source dataset. A generator that is the only one drawn from its dataset
is, in this sample, indistinguishable from that dataset.

In [ ]:
f_idx = index[fake]
ct = pd.crosstab(f_idx.generator, f_idx.origin_dataset)
order = f_idx.drop_duplicates("generator").sort_values("release_date").generator
ct = ct.loc[order]
fig, ax = plt.subplots(figsize=(8, 9))
ax.imshow(ct.values > 0, cmap=LinearSegmentedColormap.from_list("b", ["white", BLUE]),
          aspect="auto")
for (i, j), v in np.ndenumerate(ct.values):
    if v:
        ax.text(j, i, v, ha="center", va="center", fontsize=7, color="white")
ax.set_xticks(range(ct.shape[1]), ct.columns, rotation=60, ha="right")
fams = f_idx.drop_duplicates("generator").set_index("generator").generator_family
ax.set_yticks(range(ct.shape[0]), [f"{g}  [{fams[g]}]" for g in ct.index])
ax.set_title("fakes per generator x origin dataset (rows in release order)")
plt.tight_layout(); plt.show()

## 5. Inside each family, as granularity grows

For every family with at least two generators: do that family's generators separate on the
first *g* components? Each panel compares

* **centred** (blue): the first *g* PCs of the pooled family-centred PCA. These axes were chosen
  for within-family variance shared across families.
* **raw** (orange): the first *g* PCs of the raw PCA. These axes are dominated by
  between-family variance.
* **full Φ** (dashed): all scaled dimensions, no PCA. The ceiling the description itself
  allows, at this sample size.
* grey band: permutation null for the centred curve (generator labels shuffled within the
  family, neighbour graph fixed), median to 95th percentile.

Balanced accuracy, so chance = 1 / (generators in family). At 17 images per generator, one image
is 6 points of a single generator's recall. With two generators ("other"), leave-one-out k-NN
under no signal sits *below* 0.5: removing a point tilts its neighbourhood toward the other
class. The null band shows this.

How far apart blue and orange can be depends on the between-family share of variance printed
in §2. When families barely differ in mean Φ, subtracting the means hardly rotates the PCA and
the two curves coincide. Then the family-centred view is not adding information at this
granularity; the within-family structure was already the dominant variance in the raw view.

In [ ]:
WITHIN = [f for f in A.FAMILY_ORDER[1:] if index[fam == f].generator.nunique() >= 2]
G = [g for g in A.GRANULARITIES if g <= P_c.shape[1]]

def within_family(P_c, P_r, Z, label_col="generator", n_perm=N_PERM):
    out = []
    for f in WITHIN:
        sel = fam == f
        lab = index.loc[sel, label_col].to_numpy()
        if len(np.unique(lab)) < 2:
            continue
        full = A.separability(Z[sel], lab, n_perm=0, kmeans=False)["knn_acc"]
        for g in G:
            c = A.separability(P_c[sel, :g], lab, n_perm=n_perm, kmeans=True)
            r = A.separability(P_r[sel, :g], lab, n_perm=0, kmeans=True)
            out.append(dict(family=f, g=g, label=label_col, levels=c["levels"], chance=c["chance"],
                            centred=c["knn_acc"], null_mean=c["null_mean"], null_q95=c["null_q95"],
                            p=c["p"], centred_ari=c["ari"], raw=r["knn_acc"], raw_ari=r["ari"],
                            full=full))
    return pd.DataFrame(out)

wf = within_family(P_c, P_r, Z)

fig, axes = plt.subplots(1, len(WITHIN), figsize=(3.3 * len(WITHIN), 3.2), sharey=True)
for ax, f in zip(np.atleast_1d(axes), WITHIN):
    d = wf[wf.family == f]
    ax.fill_between(d.g, d.null_mean, d.null_q95, color=NULLBAND, lw=0, label="null (median-95%)")
    ax.plot(d.g, d.raw, color=ORANGE, lw=2, marker="o", ms=4, label="raw PCs")
    ax.plot(d.g, d.centred, color=BLUE, lw=2, marker="o", ms=4, label="centred PCs")
    ax.axhline(d.full.iloc[0], color=MUTED, lw=1, ls="--", label="full Φ")
    ax.axhline(d.chance.iloc[0], color=MUTED, lw=0.8, ls=":")
    ax.set_xscale("log"); ax.set_xticks(G, [str(g) for g in G])
    ax.set_title(f"{f}: {d.levels.iloc[0]} generators")
    ax.set_xlabel("g (components kept)")
np.atleast_1d(axes)[0].set_ylabel("within-family generator\nbalanced 5-NN accuracy")
np.atleast_1d(axes)[-1].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout(); plt.show()

wf.round(3)

### 5a. The same question with the confounds as labels

The same within-family separability, now with `origin_dataset` and `source_format` as the
labels, where the family has more than one level of them. If these match or beat the
generator curve, within-family structure is at least partly dataset or format structure.

In [ ]:
conf = pd.concat([within_family(P_c, P_r, Z, col, n_perm=200)
                  for col in ("origin_dataset", "source_format")], ignore_index=True)
gen = wf.assign(label="generator")
both = pd.concat([gen, conf])
pivot = both[both.g.isin([5, 20])].pivot_table(index=["family", "label"], columns="g",
                                               values=["centred", "null_q95", "chance"])
pivot.round(3)

### 5a′. Deconfounded: generators that share family *and* origin dataset

§5a can only say that generator and dataset separability are comparable; it cannot say which one
drives it. This test removes dataset identity by construction. Images are grouped into cells of
one family × one origin dataset (and, in the stricter variant, × one source format). Only
generators with at least 5 images in a cell are kept, and only cells with at least two such
generators. Neighbours are searched **only inside the cell**, and the null shuffles generator
labels **only inside the cell**. What remains separable here cannot be explained by which
dataset, or which container, an image came from. It could still be anything else the
generators' source datasets differ in (prompt or content distribution, native size, processing
before release).

The pooled null sits below the weighted "chance" because leave-one-out k-NN in small,
few-class cells is biased low under no signal. Compare against the null, not chance.

In [ ]:
strata_od = (index.generator_family + " | " + index.origin_dataset).to_numpy()
strata_odf = (index.generator_family + " | " + index.origin_dataset + " | "
              + index.source_format).to_numpy()
gen_lab = index.generator.to_numpy()

variants = [("family x origin", strata_od), ("family x origin x format", strata_odf)]

rows, cell_tables = [], {}
for name, strata in variants:
    for g in G:
        out, tab = A.stratified_separability(P_c[fake, :g], gen_lab[fake], strata[fake],
                                             n_perm=N_PERM)
        rows.append(dict(cells=name, g=g, **out))
        cell_tables[(name, g)] = tab
decon = pd.DataFrame(rows)
display(decon.round(3))
a, b = (decon[decon.cells == v[0]].knn_acc.to_numpy() for v in variants)
if np.allclose(a, b):
    print("The two variants coincide: every qualifying cell already has a single source_format, "
          "so the family x origin test holds format fixed too.")

print("per cell, family x origin, g = 20:")
cell_tables[("family x origin", 20)].round(2)

### 5b. What within-family structure looks like (date as colour only)

Each family's images on the first two raw PCs (left) and on the first two centred PCs (right).
Colour is release date on one ramp shared by every family (light = 2017, dark = 2024). Generator
centroids are labelled and joined in release order. **This is visual orientation only: no statistic
against date is computed here.** The relation to release date is the next notebook's question.

In [ ]:
dates = index.release_date
norm = Normalize(dates[fake].min().value, dates[fake].max().value)

for f in WITHIN:
    sel = fam == f
    sub = index[sel]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    for ax, P, view in [(axes[0], P_r, "raw PC1-PC2"), (axes[1], P_c, "family-centred PC1-PC2")]:
        pts = P[sel, :2]
        ax.scatter(*pts.T, c=DATE_CMAP(norm(sub.release_date.map(lambda d: d.value))), s=14,
                   linewidths=0.3, edgecolors="white")
        cent = (pd.DataFrame(pts, columns=["x", "y"]).assign(gen=sub.generator.values,
                                                              date=sub.release_date.values)
                .groupby("gen").agg(x=("x", "mean"), y=("y", "mean"), date=("date", "first"))
                .sort_values("date"))
        ax.plot(cent.x, cent.y, color=MUTED, lw=0.8, zorder=1)
        ax.scatter(cent.x, cent.y, s=150, marker="o", color="white", edgecolors=INK, lw=0.8,
                   zorder=3)
        for num, (g, r) in enumerate(cent.iterrows(), start=1):
            ax.text(r.x, r.y, str(num), ha="center", va="center", fontsize=6.5, color=INK,
                    zorder=4)
        ax.set_title(f"{f}: {view}")
        ax.set_xticks([]); ax.set_yticks([])
    key = "\n".join(f"{n:>2}  {g}  ({pd.Timestamp(d):%Y-%m})"
                    for n, (g, d) in enumerate(cent.date.items(), start=1))
    fig.text(1.0, 0.5, "centroids, release order\n" + key, fontsize=7.5, color=INK, va="center",
             family="monospace")
    sm = plt.cm.ScalarMappable(norm=norm, cmap=DATE_CMAP)
    cb = fig.colorbar(sm, ax=axes, shrink=0.8, pad=0.01)
    ticks = pd.to_datetime([f"{y}-01-01" for y in range(2017, 2025)])
    cb.set_ticks([t.value for t in ticks], labels=[str(t.year) for t in ticks])
    cb.outline.set_visible(False)
    plt.show()

### 5c. Reference: within-group structure that is *not* temporal

Authentic images have no generator and no date, but they do come from three sources (COCO,
LAION, RAISE) with different capture and processing pipelines. Centred on their own mean and
projected onto the fake-fitted centred axes, how well do the sources separate? This puts a
scale on "within-group structure": the processing differences between sources, which are
present everywhere in this sample, set a non-temporal benchmark.

In [ ]:
auth = ~fake
lab = index.loc[auth, "origin_dataset"].to_numpy()
ref = pd.DataFrame([dict(g=g, **A.separability(P_c[auth, :g], lab, n_perm=200, kmeans=False))
                    for g in G])
ref[["g", "levels", "chance", "knn_acc", "null_q95", "p"]].round(3)

## 6. Do the conclusions hold across settings?

The same summary for every extracted config: K ∈ {1, 2, 4, 8, 16} (B = 11, L = 3), B ∈ {5, 7, 11},
L ∈ {1, 2, 3}, plus the authors' released FRE (K = 8, M = 15). For the paper default, the
scaling choice is varied too. Permutation counts are lower here (200) to keep the sweep quick.
Reported per config:

* family / generator / origin_dataset balanced 5-NN accuracy on the raw view (first 50 PCs);
* per family, within-family generator accuracy on the centred view at g = 5 and g = 20, as
  *excess over the null mean*. Positive = generators separate beyond what shuffled labels give;
* the same for the deconfounded test of §5a′ (generators sharing family and origin dataset),
  column "same-origin".

In [ ]:
def summarise(config, scaling="robust", n_perm=200):
    idx, Xs, _, m = A.load(EXP, config)
    assert (idx.image_id == index.image_id).all()   # every config shares images.parquet
    Zs = A.scale(Xs, scaling)
    Pc, _ = A.centred_pca(A.family_centre(Zs, fam), fake, n_components=50)
    Pr, _ = A.centred_pca(Zs - Zs[fake].mean(0), fake, n_components=50)
    row = dict(config=config, scaling=scaling, dim=Xs.shape[1])
    for name, sel, lab in [("family", np.ones(len(idx), bool), fam),
                           ("generator", fake, idx.generator.to_numpy()),
                           ("origin", fake, idx.origin_dataset.to_numpy())]:
        s = A.separability(Pr[sel], lab[sel], n_perm=n_perm, kmeans=False)
        row[f"raw {name} acc"] = s["knn_acc"]
        row[f"raw {name} excess"] = s["knn_acc"] - s["null_mean"]
    strata = (idx.generator_family + " | " + idx.origin_dataset).to_numpy()
    for g in (5, 20):
        out, _ = A.stratified_separability(Pc[fake, :g], idx.generator[fake].to_numpy(),
                                           strata[fake], n_perm=n_perm)
        row[f"same-origin g={g}"] = out["knn_acc"] - out["null_mean"]
        row[f"same-origin g={g} p"] = out["p"]
    for f in WITHIN:
        sel = fam == f
        for g in (5, 20):
            s = A.separability(Pc[sel, :g], idx.generator[sel].to_numpy(), n_perm=n_perm,
                               kmeans=False)
            row[f"{f} g={g}"] = s["knn_acc"] - s["null_mean"]
            row[f"{f} g={g} p"] = s["p"]
    return row

runs = [summarise(c) for c in A.configs(EXP)]
runs += [summarise(CONFIG, s) for s in ("standard", "none")]
sweep = pd.DataFrame(runs).set_index(["config", "scaling"])
sweep.round(3)

In [ ]:
excess_cols = [c for c in sweep.columns if c.endswith(("g=5", "g=20"))]
H = sweep[excess_cols]
fig, ax = plt.subplots(figsize=(1.0 + 0.75 * len(excess_cols), 0.42 * len(H) + 1.2))
lim = np.abs(H.values).max()
# diverging: below the null (orange) / at the null (neutral grey) / above it (blue)
im = ax.imshow(H.values, cmap=LinearSegmentedColormap.from_list(
                   "e", ["#a8401a", ORANGE, "#e9e8e4", BLUE, "#0b2a55"]),
               vmin=-lim, vmax=lim, aspect="auto")
for (i, j), v in np.ndenumerate(H.values):
    p = sweep[excess_cols[j] + " p"].iloc[i]
    ax.text(j, i, f"{v:+.2f}{'*' if p < 0.05 else ''}", ha="center", va="center", fontsize=7,
            color="white" if abs(v) > lim * 0.55 else INK)
ax.set_xticks(range(len(excess_cols)), excess_cols, rotation=45, ha="right")
ax.set_yticks(range(len(H)), [f"{c} [{s}]" for c, s in H.index])
ax.set_title("within-family generator accuracy, excess over permutation null "
             "(centred view; * = p < 0.05)")
fig.colorbar(im, ax=ax, shrink=0.7, pad=0.01).outline.set_visible(False)
plt.tight_layout(); plt.show()

# Does the *pattern* of within-family separability agree across settings?
print("Spearman correlation of per-cell excess with the paper default, across settings:")
print(H.T.corr(method="spearman")[(CONFIG, "robust")].round(2).to_string())

## 7. Reading (local sample, 2026-10-10; logged as DECISIONS.md R-001)

Numbers are for the paper default `K8_M11_B11_L3` with robust scaling unless stated. "Across settings"
means the 12 rows of §6: 10 configs, plus 2 alternative scalings of the default.

**1. Family structure is present but weak.** Only **2.2 %** of the fake rows' variance lies between
family means. Family balanced 5-NN on the raw view is 0.30 among fakes (6 families, chance 0.17,
null q95 0.19), and KMeans recovers almost none of it (NMI 0.05). Across settings, the
family excess over the null is +0.04 to +0.13: always positive, never large. The tight
per-generator clusters of the paper's t-SNE (Fig. 5) do **not** reproduce here. [I] Candidate
reasons, untested: every image here was re-encoded at JPEG q95 (the paper's JPEG test [S: §7, Tab. 6]
covers real-vs-fake AUC only, not attribution or clustering); 200 × 200 crops instead of 1024; and
the paper's sources may be as dataset-confounded as ours (see 2).

**2. Φ tracks where an image came from at least as much as what made it.** The three authentic
sources separate at 0.70 (chance 0.33), more strongly than generator families do. Among fakes,
origin dataset (0.165, chance 0.062) and native-size bucket (0.49, chance 0.33) separate about as
well as generator (0.116, chance 0.028). Any claim about generators has to be made inside a fixed
origin dataset.

**3. Raw vs family-centred barely differ, because there is little family mean to remove.** At 2.2 %
between-family variance, centring hardly rotates the PCA. The raw and centred curves of §5
coincide, and so do the PC1–PC2 scatters. On this sample, the family-centred view adds no
information over the raw one. It will only matter where families differ more in mean Φ.

**4. Within-family generator structure survives deconfounding, and that conclusion holds in every
setting.** Restrict to generators that share family **and** origin dataset (12 cells, 484 images; every
qualifying cell has a single source format, so format is fixed too), search neighbours only within a cell,
and shuffle only within a cell. Generator 5-NN is then 0.33 at g = 5 and 0.38 at g = 10–50,
against a null of 0.19 (p = 0.002). Across all 12 settings the excess is **+0.09 to +0.20** at
g = 5 and **+0.11 to +0.20** at g = 20, every one at the smallest attainable p. Clearest cells:
inpainting in Artifact (LaMa / MAT / SN-PatchGAN, 0.59 vs chance 0.33), GANs in Artifact (12
generators, 0.21 vs 0.08), FLUX Dev vs Schnell (same dataset, released a day apart, 0.62 vs 0.50).

**5. Granularity.** Within-family separability rises from g = 1–2 to g ≈ 10 and then plateaus near
the full-Φ ceiling. The generator-distinguishing variation sits in roughly the first 10–20
pooled within-family components.

**6. What does not hold across settings: the per-family detail.** How strongly each individual
family separates changes with K, B and L. Spearman correlation of the per-cell pattern with the
default ranges from −0.02 (K = 16) to 0.76 (L = 2), against 0.94–0.95 for the two other
scalings, so scaling does not matter but the description's shape does. A single scale (L = 1)
is weakest, matching the paper's own ablation [S: Tab. 5, "No multi-scale" 0.956 vs 0.986 AUC].
K = 16 does not improve on K = 8, plausibly because the joint fit is worse conditioned (median
‖Φ‖ 382 vs 129). Do not read per-family rankings off any one config.

**For the time hypothesis:** its precondition holds weakly but robustly. Self-descriptions do
vary between generators of the same family, even with dataset and format held fixed. Whether
that variation is *time-specific* is untested here. In the diffusion PC1–PC2 view the 2023–24
generators sit apart from the 2021–22 ones, but FLUX all comes from one dataset (SFHQ-T2I), so
this is not evidence yet; it is the next notebook's question.

**Caveats.** 17 images per generator; every image JPEG-q95 re-encoded; 200 × 200 crops; FREs
trained on 306 authentic crops. The server run (stratified 36k, same frozen FREs) is the test
of whether 4–6 survive more data.